In [2]:
# Import statements
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.model_selection import cross_val_score, GridSearchCV
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score
from pandas.tseries.offsets import DateOffset

from lightgbm import LGBMRegressor

In [3]:
# Uploading the dataframe
df1 = pd.read_csv('idx_cleaned_data1.csv')
df2 = pd.read_csv('idx_cleaned_data2.csv', low_memory=False)
df3 = pd.read_csv('idx_cleaned_data3.csv', low_memory=False)
df = pd.merge(df1, df2, how='outer')
df = pd.merge(df, df3, how='outer')

In [4]:
# Removing any potential outliers within the ClosePrice variable
Q1 = df['ClosePrice'].quantile(0.005)
Q3 = df['ClosePrice'].quantile(0.995)

df = df[(df['ClosePrice'] >= Q1) & (df['ClosePrice'] <= Q3)]

In [5]:
# Creating the train/test split (time-based)
df['CloseDate'] = pd.to_datetime(df['CloseDate'], yearfirst=True)

last_date  = df['CloseDate'].max()
test_date  = last_date - DateOffset(months=1)
train_date = test_date - DateOffset(months=6)

train_df = df[(df['CloseDate'] > train_date) & (df['CloseDate'] < test_date)]
test_df  = df[df['CloseDate'] >= test_date]
test_df  = test_df[
    (test_df['ClosePrice'] > test_df['ClosePrice'].quantile(0.05)) &
    (test_df['ClosePrice'] < test_df['ClosePrice'].quantile(0.95))
]

train_y = train_df['ClosePrice']
test_y  = test_df['ClosePrice']

In [6]:
# Dropping columns to improve the df
drop_cols = [
    'Flooring', 'Levels', 'UnparsedAddress', 'PostalCode', 'City',
    'StreetNumberNumeric', 'PropertyType', 'PropertySubType',
    'ClosePrice', 'CloseDate', 'DaysOnMarket', 'BuyerOfficeName',
    'BuyerAgentMlsId', 'BuyerAgentFirstName', 'BuyerAgentLastName',
    'BuyerAgentAOR', 'BuyerOfficeAOR', 'ContractStatusChangeDate',
    'PurchaseContractDate', 'MLSAreaMajor', 'CountyOrParish',
    'ElementarySchool', 'SubdivisionName', 'ListingContractDate',
    'HighSchool', 'HighSchoolDistrict', 'StateOrProvince',
    'MiddleOrJuniorSchool'
]
train_X = train_df.drop(columns=drop_cols)
test_X  = test_df.drop(columns=drop_cols)

In [7]:
# Keep only numeric columns (same logic as original)
num_cols = train_df.select_dtypes(include='number').columns
nf_train_df = train_df.copy()

In [8]:
# ── NEW FEATURES (from original notebook) ──────────────────────────────────
# bath-to-bedroom ratio
nf_train_df['bbratio'] = (
    nf_train_df['BathroomsTotalInteger'] /
    nf_train_df['BedroomsTotal'].replace(0, 1)
).fillna(0)

# property age at contract date
nf_train_df['age'] = (
    pd.to_datetime(nf_train_df['ContractStatusChangeDate']).dt.year
    - nf_train_df['YearBuilt']
)

# Mirror the same new features onto test_df
test_df = test_df.copy()
test_df['bbratio'] = (
    test_df['BathroomsTotalInteger'] /
    test_df['BedroomsTotal'].replace(0, 1)
).fillna(0)
test_df['age'] = (
    pd.to_datetime(test_df['ContractStatusChangeDate']).dt.year
    - test_df['YearBuilt']
)

In [9]:
# Refresh num_cols after adding new features
num_cols    = nf_train_df.select_dtypes(include='number').columns
nf_train_df = nf_train_df[num_cols]

In [10]:
# Prepare training and test arrays — use all numeric columns
X_train = nf_train_df.drop(columns=['ClosePrice'])
y_train = nf_train_df['ClosePrice']

# Align test columns with training columns (fill any missing with 0)
test_num_cols = num_cols  # same set used for training
X_test  = test_df[test_num_cols].drop(columns=['ClosePrice'], errors='ignore')
X_test  = X_test.reindex(columns=X_train.columns, fill_value=0)
y_test  = test_df['ClosePrice']

print(f"Train shape: {X_train.shape}  |  Test shape: {X_test.shape}")

Train shape: (99165, 19)  |  Test shape: (20375, 19)


In [11]:
param_grid = {
    'n_estimators':    [100, 300, 500],
    'max_depth':       [3, 5, 7],
    'learning_rate':   [0.05, 0.1, 0.2],
    'subsample':       [0.7, 1.0],
    'colsample_bytree':[0.7, 1.0],
}

lgbm_base = LGBMRegressor(
    objective='regression',
    random_state=42,
    n_jobs=1
)

grid_search = GridSearchCV(
    estimator=lgbm_base,
    param_grid=param_grid,
    scoring='r2',
    cv=3,
    n_jobs=1
)

grid_search.fit(X_train, y_train)

Streaming output truncated to the last 5000 lines.
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with posit

GridSearchCV(cv=3,
             estimator=LGBMRegressor(n_jobs=1, objective='regression',
                                     random_state=42),
             n_jobs=1,
             param_grid={'colsample_bytree': [0.7, 1.0],
                         'learning_rate': [0.05, 0.1, 0.2],
                         'max_depth': [3, 5, 7],
                         'n_estimators': [100, 300, 500],
                         'subsample': [0.7, 1.0]},
             scoring='r2')

In [12]:
print("\nBest parameters:", grid_search.best_params_)
print(f"Best CV R²:       {grid_search.best_score_:.4f}")

best_xgb = grid_search.best_estimator_
y_pred   = best_xgb.predict(X_test)

r2   = r2_score(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))

print(f"Test R²:   {r2:.4f}")
print(f"Test RMSE: {rmse:,.2f}")



Best parameters: {'colsample_bytree': 1.0, 'learning_rate': 0.05, 'max_depth': 7, 'n_estimators': 500, 'subsample': 0.7}
Best CV R²:       0.5375
Test R²:   0.4004
Test RMSE: 438,564.81


In [13]:
# Calculate MAPE and MdAPE
def mean_absolute_percentage_error(y_true, y_pred):
    y_true, y_pred = np.array(y_true), np.array(y_pred)
    return np.mean(np.abs((y_true - y_pred) / y_true)) * 100

def median_absolute_percentage_error(y_true, y_pred):
    y_true, y_pred = np.array(y_true), np.array(y_pred)
    return np.median(np.abs((y_true - y_pred) / y_true)) * 100

mape = mean_absolute_percentage_error(y_test, y_pred)
md_ape = median_absolute_percentage_error(y_test, y_pred)

print(f"Test MAPE:   {mape:.2f}%")
print(f"Test MdAPE:  {md_ape:.2f}%")

Test MAPE:   2127.63%
Test MdAPE:  25.22%


In [14]:
# Summarize insights on price bands
results_df = pd.DataFrame({'actual': y_test, 'predicted': y_pred})
results_df['error'] = np.abs(results_df['actual'] - results_df['predicted'])
results_df['percentage_error'] = np.abs((results_df['actual'] - results_df['predicted']) / results_df['actual']) * 100


In [15]:
# Define price bands
price_bins = [
    0, 500000, 750000, 1000000, 1500000, 2000000, np.inf
]
price_labels = [
    ' < $500k', '$500k - $750k', '$750k - $1M',
    '$1M - $1.5M', '$1.5M - $2M', '> $2M'
]
results_df['price_band'] = pd.cut(
    results_df['actual'],
    bins=price_bins,
    labels=price_labels,
    right=False
)

In [16]:
# Analyze performance by price band
metrics_summary = results_df.groupby('price_band').agg(
    mean_actual=('actual', 'mean'),
    mean_predicted=('predicted', 'mean'),
    mean_abs_error=('error', 'mean'),
    mean_percentage_error=('percentage_error', 'mean'),
    median_percentage_error=('percentage_error', 'median'),
    count=('actual', 'count')
).reset_index()

print("\nModel Performance by Price Band:")
print(metrics_summary.round(2))


Model Performance by Price Band:
      price_band  mean_actual  mean_predicted  mean_abs_error  \
0        < $500k    146385.08       462222.96       355302.18   
1  $500k - $750k    621667.02       607910.21       126339.01   
2    $750k - $1M    858589.93       799509.48       169807.51   
3    $1M - $1.5M   1222369.82      1088137.61       274841.46   
4    $1.5M - $2M   1707457.87      1478092.21       399667.17   
5          > $2M   2197365.03      1773602.03       565782.55   

   mean_percentage_error  median_percentage_error  count  
0                5701.10                  1974.09   7556  
1                  20.36                    15.02   4215  
2                  19.78                    14.87   3385  
3                  22.35                    18.89   3133  
4                  23.44                    20.24   1493  
5                  25.70                    22.32    593  


/tmp/ipykernel_6600/2577042217.py:2: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  band_performance = results_df.groupby('price_band').agg(


In [17]:
# Save the band_performance DataFrame to a CSV file
metrics_summary.to_csv('metrics_summary.csv', index=False)
print("metrics_summary.csv has been saved.")

metrics_summary.csv has been saved.
